# 01 - downloading the raw data

this notebook downloads all the raw tlc trip parquet files from the nyc taxi and limousine commission website. we need yellow, green, fhvhv (uber/lyft) and fhv from 2022 through to early 2026.

it skips files that are already on disk so you can safely re-run if something gets interrupted.

In [ ]:
import os
import time
import urllib.request

BASE_URL = 'https://d37ci6vzurychx.cloudfront.net/trip-data'

TYPES = {
    'yellow': 'yellow_tripdata',
    'green':  'green_tripdata',
    'fhvhv':  'fhvhv_tripdata',
    'fhv':    'fhv_tripdata',
}

RAW_DIR = os.path.join(os.path.dirname(os.getcwd()), 'raw')
RETRY_DELAYS = [10, 30, 60]

the download function retries up to 3 times with increasing delays if something fails. for 2026 we only go up to february since the data isn't all there yet.

In [ ]:
def months_for_year(year):
    limit = 3 if year == 2026 else 13
    return range(1, limit)


def download_file(url, dest):
    for attempt, delay in enumerate([0] + RETRY_DELAYS):
        if delay:
            print(f'    retry {attempt}/{len(RETRY_DELAYS)} in {delay}s...', flush=True)
            time.sleep(delay)
        try:
            urllib.request.urlretrieve(url, dest)
            return True
        except Exception as e:
            err = str(e).split('\n')[0]
            if attempt == len(RETRY_DELAYS):
                print(f'skip ({err})', flush=True)
                if os.path.exists(dest):
                    os.remove(dest)
                return False
            print(f'error ({err})', end=' ', flush=True)
    return False

now loop over every cab type and every month. files go into `raw/<cab_type>/` subfolders.

In [ ]:
for cab, prefix in TYPES.items():
    type_dir = os.path.join(RAW_DIR, cab)
    os.makedirs(type_dir, exist_ok=True)

    for year in range(2022, 2027):
        for month in months_for_year(year):
            filename = f'{prefix}_{year}-{month:02d}.parquet'
            dest = os.path.join(type_dir, filename)

            if os.path.exists(dest):
                print(f'  [skip]  {filename}')
                continue

            url = f'{BASE_URL}/{filename}'
            print(f'  [fetch] {filename} ...', end=' ', flush=True)
            if download_file(url, dest):
                size_mb = os.path.getsize(dest) / 1e6
                print(f'{size_mb:.0f} mb')